In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import math
from sklearn.metrics import (
    balanced_accuracy_score,
    average_precision_score,
    roc_auc_score,
)

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from scipy.stats import pearsonr
from sklearn.metrics import mean_squared_error

import wandb

# Change working directory to the notebook's directory
notebook_dir = os.path.dirname(os.path.abspath("frozen_mlp_age.ipynb"))
os.chdir(notebook_dir)

# -------------------
# Class balance (kept from your code)
# -------------------
_NUM_POS = 0.29
_NUM_NEG = 0.71
_POS_WEIGHT = torch.tensor([_NUM_NEG / _NUM_POS])  # e.g., ~2.45; keep your value

# -------------------
# Small cosine schedule helper (0-indexed epochs)
# -------------------
def cosine_schedule(start_val, end_val, total_epochs, warmup_epochs=0):
    def schedule(epoch):
        if epoch < warmup_epochs:
            return start_val + (end_val - start_val) * epoch / warmup_epochs
        progress = (epoch - warmup_epochs) / (total_epochs - warmup_epochs)
        cosine_decay = 0.5 * (1 + math.cos(math.pi * progress))
        return end_val + (start_val - end_val) * cosine_decay
    return schedule

# -------------------
# Model
# -------------------
class MLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.Linear(512, 1)  # regression output
        )
    def forward(self, x):
        return self.net(x)

# -------------------
# Utilities
# -------------------
def build_Xy(ids, features_dict, class_map):
    """
    Filters subjects to those present in both `features_dict` and `class_map`,
    preserving the order in `ids`. Returns X, y, subj_ids_kept.
    """
    kept_ids = [sid for sid in ids if sid.split('_')[0] in class_map and sid in features_dict]
    if len(kept_ids) == 0:
        raise ValueError("No overlapping subjects found between ids, features, and labels.")
    X = np.vstack([features_dict[sid] for sid in kept_ids])
    y = np.array([class_map[sid.split('_')[0]] for sid in kept_ids])
    X[np.isnan(X)]=0
    return X, y, kept_ids

def normalize_like_train(X, train_mean, train_std):
    return (X - train_mean) / (train_std + 1e-8)

def save_predictions_csv(csv_path, subject_ids, y_true, y_pred):
    df = pd.DataFrame({
        "subject_id": subject_ids,
        "y_true": y_true.astype(float),
        "y_pred": y_pred.astype(float),
    })
    os.makedirs(os.path.dirname(csv_path), exist_ok=True)
    df.to_csv(csv_path, index=False)

@torch.no_grad()
def evaluate_model_regression(model, X, y, subject_ids, batch_size=64, device="cpu"):
    model.eval()
    ds = TensorDataset(torch.tensor(X, dtype=torch.float32),
                       torch.tensor(y, dtype=torch.float32).unsqueeze(1))
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False)

    preds, truths = [], []
    for xb, yb in loader:
        xb = xb.to(device)
        out = model(xb).cpu().numpy().flatten()
        preds.append(out)
        truths.append(yb.cpu().numpy().flatten())
    y_pred = np.concatenate(preds)
    y_true = np.concatenate(truths)

    # Metrics: Pearson r, p-value, MSE
    r, p = (np.nan, np.nan)
    # Guard pearsonr if constant
    if np.std(y_true) > 0 and np.std(y_pred) > 0:
        r, p = pearsonr(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)

    metrics = {"r": r, "p_value": p, "mse": mse}
    return metrics, y_true, y_pred

# -------------------
# Training loop with best-checkpoint saving
# -------------------
def train_model(
    X_train, y_train, train_ids_kept,
    X_val, y_val, val_ids_kept,
    input_dim, feat_file, project_name,
    epochs=100, batch_size=32, base_lr=1e-5,
):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    train_dataset = TensorDataset(torch.tensor(X_train, dtype=torch.float32),
                                  torch.tensor(y_train, dtype=torch.float32).unsqueeze(1))
    val_dataset = TensorDataset(torch.tensor(X_val, dtype=torch.float32),
                                torch.tensor(y_val, dtype=torch.float32).unsqueeze(1))
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=False)
    val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False, drop_last=False)

    model = MLP(input_dim=input_dim).to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=base_lr,
        betas=(0.9, 0.999),
        weight_decay=0.04
    )
    criterion = nn.MSELoss()

    wandb.init(project=project_name, reinit=True)
    wandb.config.update({"epochs": epochs, "batch_size": batch_size, "base_lr": base_lr})

    lr_schedule = cosine_schedule(start_val=5e-5, end_val=1e-6, total_epochs=epochs, warmup_epochs=40)
    wd_schedule = cosine_schedule(start_val=0.04, end_val=0.04, total_epochs=epochs, warmup_epochs=0)

    # Best-checkpoint tracking
    best_val = -float("inf")
    best_epoch = -1
    best_ckpt_path_state = None
    best_ckpt_path_full  = None
    key_val = 'bal_acc'  # choose validation metric to maximize

    run_name = wandb.run.name if wandb.run is not None else "unnamed_run"
    save_dir = wandb.run.dir if wandb.run is not None else "checkpoints"
    os.makedirs(save_dir, exist_ok=True)

    for epoch in range(epochs):
        # update LR/WD
        lr = lr_schedule(epoch)
        wd = wd_schedule(epoch)
        for pg in optimizer.param_groups:
            pg["lr"] = lr
            pg["weight_decay"] = wd

        # ---- Train ----
        model.train()
        train_loss_sum, n_train = 0.0, 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            preds = model(xb)
            loss = criterion(preds, yb)
            loss.backward()
            optimizer.step()
            bs = xb.size(0)
            train_loss_sum += loss.item() * bs
            n_train += bs
        train_loss = train_loss_sum / max(1, n_train)

        # ---- Validate ----
        model.eval()
        val_loss_sum, n_val = 0.0, 0
        preds_collect, truths_collect = [], []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                out = model(xb)
                loss = criterion(out, yb)
                bs = xb.size(0)
                val_loss_sum += loss.item() * bs
                n_val += bs
                preds_collect.append(out.cpu().numpy().flatten())
                truths_collect.append(yb.cpu().numpy().flatten())
        val_loss = val_loss_sum / max(1, n_val)

        y_pred_val = np.concatenate(preds_collect)
        y_true_val = np.concatenate(truths_collect)





        # Metrics
        r_val, p_val = (np.nan, np.nan)
        if np.std(y_true_val) > 0 and np.std(y_pred_val) > 0:
            r_val, p_val = pearsonr(y_true_val, y_pred_val)
        mse_val = mean_squared_error(y_true_val, y_pred_val)

        wandb.log({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "r": r_val,
            "p_value": p_val,
            "mse": mse_val,
            "lr": lr,
            "weight_decay": wd
        })

        # ---- Save best by r ----
        current_key_val = r_val if np.isfinite(r_val) else -np.inf
        if current_key_val > best_val:
            best_val = current_key_val
            best_epoch = epoch + 1

            best_ckpt_path_state = os.path.join(save_dir, f"{run_name}_BEST_state_dict.pth")
            best_ckpt_path_full  = os.path.join(save_dir, f"{run_name}_BEST_full_model.pth")
            torch.save(model.state_dict(), best_ckpt_path_state)
            torch.save(model, best_ckpt_path_full)

            # Save VAL predictions CSV aligned to val_ids_kept
            csv_path_val = os.path.join(save_dir, f"{run_name}_BEST_val_predictions.csv")
            save_predictions_csv(csv_path_val, val_ids_kept, y_true_val, y_pred_val)
    run_code = wandb.run.id if wandb.run is not None else "no_run_id"


    plt.subplot(1, 1, 1)
    plt.scatter(y_true_val, y_pred_val, alpha=0.7)
    plt.plot([y_true_val.min(), y_true_val.max()], [y_pred_val.min(), y_pred_val.max()], 'r--')
    plt.xlabel("Ground Truth Age (normalized)")
    plt.ylabel("Predicted Age (normalized)")

    wandb.summary["best_epoch"] = best_epoch
    wandb.summary["best_val_r"] = best_val
    wandb.finish()

    return {
        "best_epoch": best_epoch,
        "best_val_r": best_val,
        "best_state_dict": best_ckpt_path_state,
        "best_full_model": best_ckpt_path_full,
        "save_dir": save_dir,
        "run_name": run_code
    }




In [ ]:
# ---- Paths / Inputs ----
summary_rows = []
for split_num in range(5):
    _NUM_RUNS = 10
    best_val_fold = -float('inf')
    best_run_info = None
    for run_idx in range(_NUM_RUNS):
        project_name = f'frozen_age_CV_fold{split_num}'
        
        train_ids = np.loadtxt(f"../../splits/hcp/train_subject_list_agdev_{split_num}", dtype=str)
        val_ids = np.loadtxt(f"../../splits/hcp/val_subject_list_agdev_{split_num}", dtype=str)
        test_ids = np.loadtxt(f"../../splits/hcp/test_subject_list_agdev_{split_num}", dtype=str)

        # -------------------
        # Load ages
        # -------------------
        df = pd.read_csv("../../metadata/hcpagdev_metadata.csv")
        age_map = dict(zip(df["src_subject_id"], df["normalize_age"]))

        # Features
        feat_file = "../../latents/cls_hcpagdev_k8pcq4ai_300.npz"
        features_dict = np.load(feat_file, allow_pickle=True)

        # ---- Build filtered X/y with aligned subject IDs ----
        X_train_raw, y_train, train_ids_kept = build_Xy(train_ids, features_dict, age_map)
        X_val_raw,   y_val,   val_ids_kept   = build_Xy(val_ids,   features_dict, age_map)
        X_test_raw,  y_test,  test_ids_kept  = build_Xy(test_ids,  features_dict, age_map)

        # ---- Normalize (train statistics only) ----
        X_mean = X_train_raw.mean(axis=0, keepdims=True)
        X_std  = X_train_raw.std(axis=0, keepdims=True) + 1e-8

        X_train = normalize_like_train(X_train_raw, X_mean, X_std)
        X_val   = normalize_like_train(X_val_raw,   X_mean, X_std)
        X_test  = normalize_like_train(X_test_raw,  X_mean, X_std)

        # ---- Train and track best checkpoint on validation ----
        train_out = train_model(
            X_train, y_train, train_ids_kept,
            X_val,   y_val,   val_ids_kept,
            input_dim=X_train.shape[1],
            feat_file=feat_file,
            project_name=project_name,
            epochs=500,
            batch_size=32,
            base_lr=5e-5
        )

        # ---- Load best model and evaluate on TEST (save CSV, do NOT show or decide on this) ----
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        best_state_path = train_out["best_state_dict"]
        save_dir = train_out["save_dir"]
        run_name = train_out["run_name"]

        # Recreate model and load best weights
        best_model = MLP(input_dim=X_train.shape[1]).to(device)
        if best_state_path is None or not os.path.exists(best_state_path):
            raise FileNotFoundError("Best state_dict checkpoint was not saved or path not found.")
        best_model.load_state_dict(torch.load(best_state_path, map_location=device))

        # Evaluate on val and test
        val_metrics, y_true_val, y_pred_val = evaluate_model_regression(
            best_model, X_val, y_val, val_ids_kept, batch_size=64, device=device
        )
        test_metrics, y_true_test, y_pred_test = evaluate_model_regression(
            best_model, X_test, y_test, test_ids_kept, batch_size=64, device=device
        )

        # Save CSV (no plotting)
        csv_path_test = os.path.join(save_dir, f"{run_name}_TEST_predictions.csv")
        save_predictions_csv(csv_path_test, test_ids_kept, y_true_test, y_pred_test)

        # Save TXT summary (no prints)
        txt_path = os.path.join(save_dir, f"{run_name}_TEST_summary.txt")
        with open(txt_path, "w") as f:
            f.write(f"[Best @ epoch {train_out['best_epoch']}] val_r={train_out['best_val_r']:.6f}\n")
            f.write(f"TEST r={test_metrics['r']:.6f} | p={test_metrics['p_value']:.3e} | MSE={test_metrics['mse']:.6f}\n")
            f.write(f"VAL predictions CSV: {os.path.join(save_dir, f'{run_name}_BEST_val_predictions.csv')}\n")
            f.write(f"TEST predictions CSV: {csv_path_test}\n")
        # Track best run for this fold
        if train_out['best_val_r'] > best_val_fold:
            best_val_fold = train_out['best_val_r']
            best_run_info = {
                'split': split_num,
                'run_name': run_name,
                'best_val': train_out['best_val_r'],
                'best_epoch': train_out['best_epoch'],
                'val_MSE': val_metrics['mse'],
                'val_rho': val_metrics['r']
            }

    # After all runs for this fold, append best run info
    summary_rows.append(best_run_info)


# Show summary table
import pandas as pd
summary_df = pd.DataFrame(summary_rows)
display(summary_df)
